Copyright (c) Microsoft Corporation. All rights reserved.  
Licensed under the MIT License.

![Impressions](https://PixelServer20190423114238.azurewebsites.net/api/impressions/NotebookVM/how-to-use-azureml/machine-learning-pipelines/intro-to-pipelines/aml-pipelines-with-automated-machine-learning-step.png)

# Azure Machine Learning Pipeline with AutoMLStep (Udacity Course 2)
This notebook demonstrates the use of AutoMLStep in Azure Machine Learning Pipeline.

## Introduction
In this example we showcase how you can use AzureML Dataset to load data for AutoML via AML Pipeline. 

If you are using an Azure Machine Learning Notebook VM, you are all set. Otherwise, make sure you have executed the [configuration](https://aka.ms/pl-config) before running this notebook.

In this notebook you will learn how to:
1. Create an `Experiment` in an existing `Workspace`.
2. Create or Attach existing AmlCompute to a workspace.
3. Define data loading in a `TabularDataset`.
4. Configure AutoML using `AutoMLConfig`.
5. Use AutoMLStep
6. Train the model using AmlCompute
7. Explore the results.
8. Test the best fitted model.

## Azure Machine Learning and Pipeline SDK-specific imports

In [1]:
import logging
import os
import csv

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from sklearn import datasets
import pkg_resources

import azureml.core
from azureml.core.experiment import Experiment
from azureml.core.workspace import Workspace
from azureml.train.automl import AutoMLConfig
from azureml.core.dataset import Dataset

from azureml.pipeline.steps import AutoMLStep

# Check core SDK version number
print("SDK version:", azureml.core.VERSION)

SDK version: 1.60.0


## Initialize Workspace
Initialize a workspace object from persisted configuration. Make sure the config file is present at .\config.json

In [2]:
ws = Workspace.from_config()
print(ws.name, ws.resource_group, ws.location, ws.subscription_id, sep = '\n')

quick-starts-ws-295275
aml-quickstarts-295275
westus2
d4ad7261-832d-46b2-b093-22156001df5b


## Create an Azure ML experiment
Let's create an experiment named "automlstep-classification" and a folder to hold the training scripts. The script runs will be recorded under the experiment in Azure.

The best practice is to use separate folders for scripts and its dependent files for each step and specify that folder as the `source_directory` for the step. This helps reduce the size of the snapshot created for the step (only the specific folder is snapshotted). Since changes in any files in the `source_directory` would trigger a re-upload of the snapshot, this helps keep the reuse of the step when there are no changes in the `source_directory` of the step.

*Udacity Note:* There is no need to create an Azure ML experiment, this needs to re-use the experiment that was already created


In [3]:
# Choose a name for the run history container in the workspace.
# NOTE: update these to match your existing experiment name
experiment_name = 'automl-bank'
project_folder = './pipeline-project'

experiment = Experiment(ws, experiment_name)
experiment

Name,Workspace,Report Page,Docs Page
automl-bank,quick-starts-ws-295275,Link to Azure Machine Learning studio,Link to Documentation


### Create or Attach an AmlCompute cluster
You will need to create a [compute target](https://docs.microsoft.com/azure/machine-learning/service/concept-azure-machine-learning-architecture#compute-target) for your AutoML run. In this tutorial, you get the default `AmlCompute` as your training compute resource.

**Udacity Note** There is no need to create a new compute target, it can re-use the previous cluster

In [4]:
from azureml.core.compute import AmlCompute
from azureml.core.compute import ComputeTarget
from azureml.core.compute_target import ComputeTargetException

# NOTE: update the cluster name to match the existing cluster
# Choose a name for your CPU cluster
amlcompute_cluster_name = "automlcluster"

# Verify that cluster does not exist already
try:
    compute_target = ComputeTarget(workspace=ws, name=amlcompute_cluster_name)
    print('Found existing cluster, use it.')
except ComputeTargetException:
    compute_config = AmlCompute.provisioning_configuration(vm_size='STANDARD_D2_V2',# for GPU, use "STANDARD_NC6"
                                                           #vm_priority = 'lowpriority', # optional
                                                           max_nodes=4)
    compute_target = ComputeTarget.create(ws, amlcompute_cluster_name, compute_config)

compute_target.wait_for_completion(show_output=True, min_node_count = 1, timeout_in_minutes = 10)
# For a more detailed view of current AmlCompute status, use get_status().

Found existing cluster, use it.
Succeeded
AmlCompute wait for completion finished

Minimum number of nodes requested have been provisioned


## Data

**Udacity note:** Make sure the `key` is the same name as the dataset that is uploaded, and that the description matches. If it is hard to find or unknown, loop over the `ws.datasets.keys()` and `print()` them.
If it *isn't* found because it was deleted, it can be recreated with the link that has the CSV 

In [5]:
# Try to load the dataset from the Workspace. Otherwise, create it from the file
# NOTE: update the key to match the dataset name
found = False
key = "bankmarketing-dataset"
description_text = ""

if key in ws.datasets.keys(): 
        found = True
        dataset = ws.datasets[key] 

if not found:
        # Create AML Dataset and register it into Workspace
        example_data = 'https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv'
        dataset = Dataset.Tabular.from_delimited_files(example_data)        
        #Register Dataset in Workspace
        dataset = dataset.register(workspace=ws,
                                   name=key,
                                   description=description_text)


df = dataset.to_pandas_dataframe()
df.describe()

{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,32950.000000,32950.000000,32950.000000,32950.000000,32950.000000,32950.000000,32950.000000,32950.000000,32950.000000,32950.000000
mean,40.040212,257.335205,2.561730,962.174780,0.174780,0.076228,93.574243,-40.518680,3.615654,5166.859608
std,10.432313,257.331700,2.763646,187.646785,0.496503,1.572242,0.578636,4.623004,1.735748,72.208448
min,17.000000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.634000,4963.600000
25%,32.000000,102.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.344000,5099.100000
50%,38.000000,179.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.000000,318.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,98.000000,4918.000000,56.000000,999.000000,7.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


### Review the Dataset Result

You can peek the result of a TabularDataset at any range using `skip(i)` and `take(j).to_pandas_dataframe()`. Doing so evaluates only `j` records for all the steps in the TabularDataset, which makes it fast even against large datasets.

`TabularDataset` objects are composed of a list of transformation steps (optional).

In [6]:
dataset.take(5).to_pandas_dataframe()

{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,57,technician,married,high.school,no,no,yes,cellular,may,mon,...,1,999,1,failure,-1.8,92.893,-46.2,1.299,5099.1,no
1,55,unknown,married,unknown,unknown,yes,no,telephone,may,thu,...,2,999,0,nonexistent,1.1,93.994,-36.4,4.860,5191.0,no
2,33,blue-collar,married,basic.9y,no,no,no,cellular,may,fri,...,1,999,1,failure,-1.8,92.893,-46.2,1.313,5099.1,no
3,36,admin.,married,high.school,no,no,no,telephone,jun,fri,...,4,999,0,nonexistent,1.4,94.465,-41.8,4.967,5228.1,no
4,27,housemaid,married,high.school,no,yes,no,cellular,jul,fri,...,2,999,0,nonexistent,1.4,93.918,-42.7,4.963,5228.1,no


## Train
This creates a general AutoML settings object.
**Udacity notes:** These inputs must match what was used when training in the portal. `label_column_name` has to be `y` for example.

In [7]:
automl_settings = {
    "experiment_timeout_minutes": 20,
    "max_concurrent_iterations": 5,
    "primary_metric" : 'AUC_weighted'
}
automl_config = AutoMLConfig(compute_target=compute_target,
                             task = "classification",
                             training_data=dataset,
                             label_column_name="y",   
                             path = project_folder,
                             enable_early_stopping= True,
                             featurization= 'auto',
                             debug_log = "automl_errors.log",
                             **automl_settings
                            )

#### Create Pipeline and AutoMLStep

You can define outputs for the AutoMLStep using TrainingOutput.

In [8]:
from azureml.pipeline.core import PipelineData, TrainingOutput

ds = ws.get_default_datastore()
metrics_output_name = 'metrics_output'
best_model_output_name = 'best_model_output'

metrics_data = PipelineData(name='metrics_data',
                           datastore=ds,
                           pipeline_output_name=metrics_output_name,
                           training_output=TrainingOutput(type='Metrics'))
model_data = PipelineData(name='model_data',
                           datastore=ds,
                           pipeline_output_name=best_model_output_name,
                           training_output=TrainingOutput(type='Model'))

Create an AutoMLStep.

In [9]:
automl_step = AutoMLStep(
    name='automl_module',
    automl_config=automl_config,
    outputs=[metrics_data, model_data],
    allow_reuse=True)

In [10]:
from azureml.pipeline.core import Pipeline
pipeline = Pipeline(
    description="pipeline_with_automlstep",
    workspace=ws,    
    steps=[automl_step])

In [11]:
pipeline_run = experiment.submit(pipeline)

Created step automl_module [306914dd][8442668e-bf39-45b8-a5d4-1d3c489ac679], (This step will run and generate new outputs)
Submitted PipelineRun 9e60ef3d-6ad2-42bc-a2c0-e166d8e2a748
Link to Azure Machine Learning Portal: https://ml.azure.com/runs/9e60ef3d-6ad2-42bc-a2c0-e166d8e2a748?wsid=/subscriptions/d4ad7261-832d-46b2-b093-22156001df5b/resourcegroups/aml-quickstarts-295275/workspaces/quick-starts-ws-295275&tid=660b3398-b80e-49d2-bc5b-ac1dc93b5254


In [12]:
#from azureml.widgets import RunDetails
#RunDetails(pipeline_run).show()

ImportError: cannot import name 'Mapping' from 'collections' (/anaconda/envs/azureml_py38/lib/python3.10/collections/__init__.py)

In [13]:
pipeline_run.wait_for_completion()

PipelineRunId: 9e60ef3d-6ad2-42bc-a2c0-e166d8e2a748
Link to Azure Machine Learning Portal: https://ml.azure.com/runs/9e60ef3d-6ad2-42bc-a2c0-e166d8e2a748?wsid=/subscriptions/d4ad7261-832d-46b2-b093-22156001df5b/resourcegroups/aml-quickstarts-295275/workspaces/quick-starts-ws-295275&tid=660b3398-b80e-49d2-bc5b-ac1dc93b5254
PipelineRun Status: Running


StepRunId: 44593c48-4cf8-4ce1-ab0c-59e26ed1b985
Link to Azure Machine Learning Portal: https://ml.azure.com/runs/44593c48-4cf8-4ce1-ab0c-59e26ed1b985?wsid=/subscriptions/d4ad7261-832d-46b2-b093-22156001df5b/resourcegroups/aml-quickstarts-295275/workspaces/quick-starts-ws-295275&tid=660b3398-b80e-49d2-bc5b-ac1dc93b5254
StepRun( automl_module ) Status: Running

StepRun(automl_module) Execution Summary
StepRun( automl_module ) Status: Finished

Warnings:
No scores improved over last 10 iterations, so experiment stopped early. This early stopping behavior can be disabled by setting enable_early_stopping = False in AutoMLConfig for notebook/py

'Finished'

## Examine Results

### Retrieve the metrics of all child runs
Outputs of above run can be used as inputs of other steps in pipeline. In this tutorial, we will examine the outputs by retrieve output data and running some tests.

In [14]:
metrics_output = pipeline_run.get_pipeline_output(metrics_output_name)
num_file_downloaded = metrics_output.download('.', show_progress=True)

Downloaded azureml/44593c48-4cf8-4ce1-ab0c-59e26ed1b985/metrics_data, 1 files out of an estimated total of 1


In [15]:
import json
with open(metrics_output._path_on_datastore) as f:
    metrics_output_result = f.read()
    
deserialized_metrics_output = json.loads(metrics_output_result)
df = pd.DataFrame(deserialized_metrics_output)
df

,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_5,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_4,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_7,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_1,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_2,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_8,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_3,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_0,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_6,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_10,...,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_45,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_41,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_38,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_40,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_43,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_46,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_48,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_53,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_44,44593c48-4cf8-4ce1-ab0c-59e26ed1b985_52
AUC_weighted,[0.9388252597495217],[0.9225368484033442],[0.9290011799639528],[0.942328104073932],[0.8958056634564979],[0.941632999720291],[0.8782909787402727],[0.9446537630106308],[0.9247722039763119],[0.9237121814143637],...,[0.930295528177428],[0.9349838935846638],[0.927933284801064],[0.5],[0.9434728728695352],[0.9441721450707331],[0.9378096942281794],[0.9460504550363342],[0.9360832791513151],[0.9473665686759395]
f1_score_micro,[0.908649468892261],[0.9071320182094081],[0.9119878603945372],[0.9116843702579667],[0.9001517450682853],[0.9104704097116844],[0.8880121396054628],[0.9101669195751139],[0.7918057663125948],[0.9083459787556905],...,[0.9113808801213961],[0.9059180576631259],[0.8030349013657057],[0.8880121396054628],[0.9144157814871017],[0.9110773899848255],[0.9080424886191198],[0.9144157814871017],[0.9089529590288316],[0.91350531107739]
matthews_correlation,[0.47077965319103915],[0.42149687086703563],[0.47805766735773875],[0.52924365161229],[0.33340661446628406],[0.5016093372462171],[0.05255258421502397],[0.5216286298277554],[0.4837997244507513],[0.444201567492641],...,[0.45490452435707335],[0.4762071323693735],[0.5071411715737993],[0.0],[0.5414091775476605],[0.5186343797812246],[0.49542618619505285],[0.5354351023345689],[0.503208212781036],[0.5071384791911904]
precision_score_micro,[0.908649468892261],[0.9071320182094081],[0.9119878603945372],[0.9116843702579667],[0.9001517450682853],[0.9104704097116844],[0.8880121396054628],[0.9101669195751139],[0.7918057663125948],[0.9083459787556905],...,[0.9113808801213961],[0.9059180576631259],[0.8030349013657057],[0.8880121396054628],[0.9144157814871017],[0.9110773899848255],[0.9080424886191198],[0.9144157814871017],[0.9089529590288316],[0.91350531107739]
recall_score_macro,[0.693976256235563],[0.6457565754741621],[0.6863829010812322],[0.7477868729473351],[0.5932768914155307],[0.7210524463412782],[0.5035523954009191],[0.7445642005975768],[0.8531718246095653],[0.6653862112783807],...,[0.6599902379748336],[0.7102002974916967],[0.8677833719553874],[0.5],[0.7516930722964099],[0.7379720550452258],[0.7232377877435643],[0.7445882814945717],[0.7284869601942773],[0.7144723412374246]
AUC_micro,[0.9779290367296751],[0.9732255383035407],[0.9758368429657296],[0.9783641467160662],[0.9671371301070044],[0.9790036405000448],[0.9632521800401124],[0.9795361989126856],[0.9027457337530308],[0.9741933909150988],...,[0.9760537532150843],[0.9769563945924413],[0.9220643776725207],[0.8880121396054628],[0.9792886172777532],[0.9793347625155142],[0.9775107821894118],[0.9801326790718451],[0.9771997393392758],[0.9804761433265557]
accuracy,[0.908649468892261],[0.9071320182094081],[0.9119878603945372],[0.9116843702579667],[0.9001517450682853],[0.9104704097116844],[0.8880121396054628],[0.9101669195751139],[0.7918057663125948],[0.9083459787556905],...,[0.9113808801213961],[0.9059180576631259],[0.8030349013657057],[0.8880121396054628],[0.9144157814871017],[0.9110773899848255],[0.9080424886191198],[0.9144157814871017],[0.9089529590288316],[0.91350531107739]
AUC_macro,[0.9388252597495217],[0.922536848403344],[0.9290011799639528],[0.942328104073932],[0.8958056634564979],[0.941632999720291],[0.8782909787402726],[0.9446537630106308],[0.92477

### Retrieve the Best Model

In [16]:
# Retrieve best model from Pipeline Run
best_model_output = pipeline_run.get_pipeline_output(best_model_output_name)
num_file_downloaded = best_model_output.download('.', show_progress=True)

Downloaded azureml/44593c48-4cf8-4ce1-ab0c-59e26ed1b985/model_data, 1 files out of an estimated total of 1


In [17]:
import pickle

with open(best_model_output._path_on_datastore, "rb" ) as f:
    best_model = pickle.load(f)
best_model

PipelineWithYTransformations(Pipeline={'memory': None,
                                       'steps': [('datatransformer',
                                                  DataTransformer(enable_dnn=False, enable_feature_sweeping=True, working_dir='/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute00/code/Users/odl_user_295275/starter_files')),
                                                 ('prefittedsoftvotingclassifier',
                                                  PreFittedSoftVotingClassifier(classification_labels=array([...nit_type': 'cpu'}), reg_alpha=0, reg_lambda=0.10416666666666667, subsample=0.7, tree_method='auto'))]))], flatten_transform=False, weights=[0.16666666666666666, 0.08333333333333333, 0.08333333333333333, 0.16666666666666666, 0.25, 0.08333333333333333, 0.08333333333333333, 0.08333333333333333]))],
                                       'verbose': False},
                             y_transformer={},
                             y_transformer_name='LabelEncoder')

In [18]:
best_model.steps

[('datatransformer',
  DataTransformer(enable_dnn=False, enable_feature_sweeping=True, feature_sweeping_config={}, feature_sweeping_timeout=86400, featurization_config=None, force_text_dnn=False, is_cross_validation=False, is_onnx_compatible=False, task='classification')),
 ('prefittedsoftvotingclassifier',
  PreFittedSoftVotingClassifier(classification_labels=numpy.array([0, 1]), estimators=[('33', Pipeline(memory=None, steps=[('standardscalerwrapper', StandardScalerWrapper(copy=True, with_mean=False, with_std=False)), ('xgboostclassifier', XGBoostClassifier(booster='gbtree', colsample_bytree=0.7, eta=0.4, gamma=5, max_depth=6, max_leaves=0, n_estimators=100, n_jobs=1, objective='reg:logistic', problem_info=ProblemInfo(gpu_training_param_dict={'processing_unit_type': 'cpu'}), random_state=0, reg_alpha=1.7708333333333335, reg_lambda=1.5625, subsample=0.5, tree_method='auto'))], verbose=False)), ('0', Pipeline(memory=None, steps=[('maxabsscaler', MaxAbsScaler(copy=True)), ('lightgbmclas

### Test the Model
#### Load Test Data
For the test data, it should have the same preparation step as the train data. Otherwise it might get failed at the preprocessing step.

In [20]:
#dataset_test = Dataset.Tabular.from_delimited_files(path='https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv')
from azureml.core import Workspace, Dataset

subscription_id = "d4ad7261-832d-46b2-b093-22156001df5b"
resource_group = "aml-quickstarts-295275"
workspace_name = "quick-starts-ws-295275"

workspace = Workspace(subscription_id, resource_group, workspace_name)

dataset_test = Dataset.get_by_name(workspace, name="bankmarketing-dataset")
df_test = dataset_test.to_pandas_dataframe()
df_test = df_test[pd.notnull(df_test['y'])]

y_test = df_test['y']
X_test = df_test.drop(['y'], axis=1)

{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


#### Testing Our Best Fitted Model

We will use confusion matrix to see how our model works.

In [21]:
from sklearn.metrics import confusion_matrix
ypred = best_model.predict(X_test)
cm = confusion_matrix(y_test, ypred)

In [22]:
# Visualize the confusion matrix
pd.DataFrame(cm).style.background_gradient(cmap='Blues', low=0, high=0.9)

,0,1
0,28891,367
1,1410,2282


## Publish and run from REST endpoint

Run the following code to publish the pipeline to your workspace. In your workspace in the portal, you can see metadata for the pipeline including run history and durations. You can also run the pipeline manually from the portal.

Additionally, publishing the pipeline enables a REST endpoint to rerun the pipeline from any HTTP library on any platform.


In [23]:
published_pipeline = pipeline_run.publish_pipeline(
    name="Bankmarketing Train", description="Training bankmarketing pipeline", version="1.0")

published_pipeline


Name,Id,Status,Endpoint
Bankmarketing Train,080017e9-7a5d-4dc6-b731-0c107d359efa,Active,REST Endpoint


Authenticate once again, to retrieve the `auth_header` so that the endpoint can be used

In [24]:
from azureml.core.authentication import InteractiveLoginAuthentication

interactive_auth = InteractiveLoginAuthentication()
auth_header = interactive_auth.get_authentication_header()



Get the REST url from the endpoint property of the published pipeline object. You can also find the REST url in your workspace in the portal. Build an HTTP POST request to the endpoint, specifying your authentication header. Additionally, add a JSON payload object with the experiment name and the batch size parameter. As a reminder, the process_count_per_node is passed through to ParallelRunStep because you defined it is defined as a PipelineParameter object in the step configuration.

Make the request to trigger the run. Access the Id key from the response dict to get the value of the run id.


In [25]:
import requests

rest_endpoint = published_pipeline.endpoint
response = requests.post(rest_endpoint, 
                         headers=auth_header, 
                         json={"ExperimentName": "pipeline-rest-endpoint"}
                        )

In [26]:
try:
    response.raise_for_status()
except Exception:    
    raise Exception("Received bad response from the endpoint: {}\n"
                    "Response Code: {}\n"
                    "Headers: {}\n"
                    "Content: {}".format(rest_endpoint, response.status_code, response.headers, response.content))

run_id = response.json().get('Id')
print('Submitted pipeline run: ', run_id)

Submitted pipeline run:  650944dc-5240-4524-9edb-73907bf08a72


Use the run id to monitor the status of the new run. This will take another 10-15 min to run and will look similar to the previous pipeline run, so if you don't need to see another pipeline run, you can skip watching the full output.